# 🧪 04. Pré-processamento e Pipelines

**Objetivo:** Preparar os dados numéricos e categóricos para o modelo de Machine Learning, garantindo que não haja vazamento de dados (Data Leakage).

**Etapas:**
1.  **Carregar:** Ler o dicionário de dados separado (Treino/Validação) do passo anterior.
2.  **Identificar Tipos:** Separar o que é numérico (Lags, Rolling) do que é categórico (Loja, Item).
3.  **Pipeline de Transformação:**
    * *Categóricos:* Ordinal Encoding ou One-Hot (para Árvores, Ordinal costuma ser suficiente e mais leve).
    * *Numéricos:* StandardScaler (opcional para árvores, mas recomendável).
4.  **Salvar:** Exportar os dados prontos para o treinamento.

## 🛠️ Imports

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


In [2]:
import pandas as pd
import pickle
import joblib
import os
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from src.tools import load_config


### Configurações gerais

In [3]:
DATA_PATH = '../data/processed/dados_modelagem.pkl'
PATH_PROCESS = '../data/processed'
PATH_MODELS = '../models'
DATA_MODELS = '../models/preprocessor.pkl'
DATA_PROCESS = '../data/processed/dados_prontos_para_treino.pkl'
DATA_XTRAIN = '../data/processed/X_train_processed.csv'
DATA_YTRAIN = '../data/processed/y_train.csv'
DATA_XVAL = '../data/processed/X_val_processed.csv'
DATA_YVAL = '../data/processed/y_val.csv'
DATA_XTEST = '../data/processed/X_test.csv'  
DATA_YTEST = '../data/processed/y_test.csv'  

In [4]:
#🚩
config = load_config()
COLUNAS_CATEGORICAS = config['colunas_categoricas']
print(f"Colunas categoricas: {COLUNAS_CATEGORICAS}")


Colunas categoricas: ['store', 'item', 'month', 'day', 'day_of_week', 'day_of_year']


## ⛁ Carga dos Dados (Pickle)

In [5]:
# Carregar o dicionário salvo no passo 03
with open(DATA_PATH, 'rb') as f:
    dados = pickle.load(f)

X_train = dados['X_train']
y_train = dados['y_train']
X_val = dados['X_val']
y_val = dados['y_val']

print(f"Treino: {X_train.shape}, Validação: {X_val.shape}")
display(X_train.head())

Treino: (730000, 32), Validação: (46000, 32)


,store,item,month,day,day_of_week,day_of_year,is_weekend,is_payday,is_holiday,days_until_holiday,...,lag_28,lag_91,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28,rolling_mean_91,rolling_std_91,sales_diff_lag1,sales_diff_lag7
0,1,1,7,2,1,183,False,1,0,67,...,2.772589,2.995732,3.115678,0.131680,3.068009,0.240008,2.938289,0.280661,-1.0,4.0
1,1,1,7,3,2,184,False,1,0,66,...,3.218876,3.218876,3.087011,0.157290,3.072216,0.235631,2.937131,0.280639,-6.0,0.0
2,1,1,7,4,3,185,False,1,0,65,...,2.302585,2.944439,3.011855,0.252135,3.048861,0.252369,2.929945,0.281714,-5.0,5.0
3,1,1,7,5,4,186,False,1,0,64,...,3.091042,2.995732,3.000860,0.239026,3.081586,0.207420,2.932961,0.283336,12.0,-9.0
4,1,1,7,6,5,187,True,1,0,63,...,3.044522,3.178054,3.000860,0.239026,3.074419,0.210524,2.931803,0.283291,-7.0,3.0


In [6]:
X_train.columns


Index(['store', 'item', 'month', 'day', 'day_of_week', 'day_of_year',
       'is_weekend', 'is_payday', 'is_holiday', 'days_until_holiday',
       'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos',
       'day_of_year_sin', 'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3',
       'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_91', 'rolling_mean_7',
       'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'rolling_mean_91',
       'rolling_std_91', 'sales_diff_lag1', 'sales_diff_lag7'],
      dtype='object')

## ⚙️ Definição de Colunas
Precisamos dizer ao modelo explicitamente quais colunas tratar como categorias.

In [7]:
# Identificar colunas
# 'store' e 'item' são inteiros, mas representam categorias. 'ano', 'mes', 'dia_da_semana' também.
# cat_cols = #['store', 'item', 'month', 'day', 'day_of_week', 'day_of_year']    #, 'days_until_holiday'    

#🚩
cat_cols = COLUNAS_CATEGORICAS   

# As demais são numéricas (Lags, Rolling means, senos/cossenos)
num_cols = [col for col in X_train.columns if col not in cat_cols]

print("Categóricas:", cat_cols)
print("Numéricas (exemplo):", num_cols[:15])

Categóricas: ['store', 'item', 'month', 'day', 'day_of_week', 'day_of_year']
Numéricas (exemplo): ['is_weekend', 'is_payday', 'is_holiday', 'days_until_holiday', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14']


In [8]:
X_train.head(5)

,store,item,month,day,day_of_week,day_of_year,is_weekend,is_payday,is_holiday,days_until_holiday,...,lag_28,lag_91,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28,rolling_mean_91,rolling_std_91,sales_diff_lag1,sales_diff_lag7
0,1,1,7,2,1,183,False,1,0,67,...,2.772589,2.995732,3.115678,0.131680,3.068009,0.240008,2.938289,0.280661,-1.0,4.0
1,1,1,7,3,2,184,False,1,0,66,...,3.218876,3.218876,3.087011,0.157290,3.072216,0.235631,2.937131,0.280639,-6.0,0.0
2,1,1,7,4,3,185,False,1,0,65,...,2.302585,2.944439,3.011855,0.252135,3.048861,0.252369,2.929945,0.281714,-5.0,5.0
3,1,1,7,5,4,186,False,1,0,64,...,3.091042,2.995732,3.000860,0.239026,3.081586,0.207420,2.932961,0.283336,12.0,-9.0
4,1,1,7,6,5,187,True,1,0,63,...,3.044522,3.178054,3.000860,0.239026,3.074419,0.210524,2.931803,0.283291,-7.0,3.0


## 🧪 Pipeline de Transformação
Usaremos `OrdinalEncoder` para as categóricas. Para modelos baseados em árvore (XGBoost, LightGBM), o Ordinal Encoding funciona muito bem e economiza memória comparado ao One-Hot Encoding (que criaria centenas de colunas para os 50 itens e 10 lojas).

In [9]:
# Pipeline Numérico: Garantir que não haja Nulos
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Previne erros se houver NaN residual
    ('scaler', StandardScaler()) # Opcional para árvores, mas ajuda na convergência
])

# Pipeline Categórico: Transformar em números ordinais (0, 1, 2...)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combinar tudo
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)


## 🔄 Fit e Transform (Treinar e Aplicar)
Regra de Ouro: `fit` apenas no Treino. `transform` no Treino e na Validação.

In [10]:
# >>>Treinar o processador APENAS nos dados de treino

print("Treinando pré-processador...")
preprocessor.fit(X_train)

# Transformar os dados
X_train_proc = preprocessor.transform(X_train)
X_val_proc = preprocessor.transform(X_val)

# Recuperar nomes das colunas para manter o DataFrame legível
# (OrdinalEncoder mantém a ordem, StandardScaler também)
cols_finais = num_cols + cat_cols

# Converter de volta para DataFrame (facilitará a análise de feature importance depois)
X_train_final = pd.DataFrame(X_train_proc, columns=cols_finais)
X_val_final = pd.DataFrame(X_val_proc, columns=cols_finais)

print("Transformação concluída.")
display(X_train_final.head())

Treinando pré-processador...
Transformação concluída.


,is_weekend,is_payday,is_holiday,days_until_holiday,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos,...,rolling_mean_91,rolling_std_91,sales_diff_lag1,sales_diff_lag7,store,item,month,day,day_of_week,day_of_year
0,-0.631243,1.431089,-0.159,0.754808,-0.702037,-1.220627,1.104023,0.883068,-0.012180,-1.416101,...,-1.678516,1.398799,-0.070757,0.280462,0.0,0.0,6.0,1.0,1.0,182.0
1,-0.631243,1.431089,-0.159,0.726423,-0.702037,-1.220627,1.377115,-0.313322,-0.036521,-1.415682,...,-1.680727,1.398241,-0.422588,-0.001596,0.0,0.0,6.0,2.0,2.0,183.0
2,-0.631243,1.431089,-0.159,0.698038,-0.702037,-1.220627,0.611929,-1.272753,-0.060851,-1.414844,...,-1.694449,1.425189,-0.352221,0.350977,0.0,0.0,6.0,3.0,3.0,184.0
3,-0.631243,1.431089,-0.159,0.669653,-0.702037,-1.220627,-0.615333,-1.272753,-0.085163,-1.413587,...,-1.688690,1.465804,0.844003,-0.636227,0.0,0.0,6.0,4.0,4.0,185.0
4,1.584177,1.431089,-0.159,0.641268,-0.702037,-1.220627,-1.380518,-0.313322,-0.109449,-1.411913,...,-1.690901,1.464700,-0.492954,0.209948,0.0,0.0,6.0,5.0,5.0,186.0


In [11]:
X_train_final.columns


Index(['is_weekend', 'is_payday', 'is_holiday', 'days_until_holiday',
       'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos',
       'day_of_year_sin', 'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3',
       'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_91', 'rolling_mean_7',
       'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'rolling_mean_91',
       'rolling_std_91', 'sales_diff_lag1', 'sales_diff_lag7', 'store', 'item',
       'month', 'day', 'day_of_week', 'day_of_year'],
      dtype='object')

## 💾 Salvar Artefatos Finais
Salvamos os DataFrames prontos e o objeto `preprocessor` (importante para processar novos dados no futuro).

In [12]:
# >>>Salva a base de X_train/y_train/x_val/y_val

X_train_final.to_csv(DATA_XTRAIN, index=False)
X_val_final.to_csv(DATA_XVAL, index=False)
y_train.to_csv(DATA_YTRAIN, index=False) 
y_val.to_csv(DATA_YVAL, index=False)

# Salva o processador (pipeline)
joblib.dump(preprocessor, DATA_MODELS)

print(f"✅ Arquivos salvos com sucesso! \n{DATA_XTRAIN} \n{DATA_XVAL} \n{DATA_YTRAIN} \n{DATA_YVAL} \n{DATA_MODELS}")

✅ Arquivos salvos com sucesso! 
../data/processed/X_train_processed.csv 
../data/processed/X_val_processed.csv 
../data/processed/y_train.csv 
../data/processed/y_val.csv 
../models/preprocessor.pkl


In [13]:
# >>>salva a base do modelo

os.makedirs(PATH_MODELS, exist_ok=True)

# Salvar dados prontos
dados_finais = {
    'X_train': X_train_final,
    'y_train': y_train,
    'X_val': X_val_final,
    'y_val': y_val
}

with open(DATA_PROCESS, 'wb') as f:
    pickle.dump(dados_finais, f)

# Salvar o Pipeline de Pré-processamento
with open(DATA_MODELS, 'wb') as f:
    pickle.dump(preprocessor, f)

print(f"✅ Dados e Processador salvos! \n{DATA_PROCESS} \n{DATA_MODELS}") 

✅ Dados e Processador salvos! 
../data/processed/dados_prontos_para_treino.pkl 
../models/preprocessor.pkl
